In [4]:
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image


# =========================
# Paths (EDIT if needed)
# =========================

#MODEL_PATH = "D:/MSCS_Research/Models/DA_Full_Fits17k_StrongAUg_CutMix/stage1_best.pt"
#MODEL_PATH = "D:/MSCS_Research/Models/DA_Full_Fits17k_StrongAUg_CutMix/stage2_best2.pt"
MODEL_PATH = "D:/MSCS_Research/Models/DA_ST1_Finetuned on DarkSkin tones only_Block4/densenet161_top3_blocks_best.pt"
#MODEL_PATH = "D:/MSCS_Research/Models/DA_ST1_Finetuned on DarkSkin tones only_Block4/densenet161_head_only_best.pt"

#MODEL_PATH = "D:/MSCS_Research/Models/DA_Output_Finetuned_Full_FIts17k_all stages_blk4/densenet161_stage2_bn_block3_4.pt"
#MODEL_PATH = "D:/MSCS_Research/Models/DA_Output_Finetuned_Full_FIts17k_all stages_blk4/densenet161_stage2_bn_block4_best.pt"
#MODEL_PATH = "D:/MSCS_Research/Models/DA_Output_Finetuned_Full_FIts17k_all stages_blk4/densenet161_Stag1_best.pt"

SUPPORT_DIR = "D:/MSCS_Research/FewShot_Evaluations/Inputs/3Way_2Shot"   # <-- folder with class subfolders, each has 5 images
QUERY_DIR   = "D:/MSCS_Research/FewShot_Evaluations/Inputs/Query3"       # <-- folder with query images


# =========================
# Device
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================
# Image transforms
# =========================
# Use same resize/crop you used in training if possible.
# These are standard ImageNet transforms for DenseNet.
IMG_SIZE = 224
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# =========================
# Utilities: load images
# =========================
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_image(path: str) -> torch.Tensor:
    img = Image.open(path).convert("RGB")
    return transform(img)


def list_images(folder: str) -> List[str]:
    p = Path(folder)
    files = [str(x) for x in p.iterdir() if x.suffix.lower() in IMG_EXTS]
    files.sort()
    return files


# =========================
# Build DenseNet161 + wrapper for features
# =========================
class DenseNetFeatureExtractor(nn.Module):
    def __init__(self, densenet: nn.Module):
        super().__init__()
        self.features = densenet.features  # backbone

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = F.relu(x, inplace=True)
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)  # [B, D]
        return x


def build_densenet161_for_loading(num_classes: int) -> nn.Module:
    """
    Build the same architecture you trained, so the classifier shape matches.
    If you don't know num_classes, we infer it from the checkpoint when possible.
    """
    m = models.densenet161(weights=None)
    in_feats = m.classifier.in_features
    m.classifier = nn.Linear(in_feats, num_classes)
    return m


def infer_num_classes_from_state_dict(state_dict: Dict[str, torch.Tensor]) -> int:
    """
    Try to infer the classifier output dim from common DenseNet keys.
    """
    # Most common key: "classifier.weight" with shape [num_classes, in_feats]
    for k in ["classifier.weight", "module.classifier.weight"]:
        if k in state_dict:
            return state_dict[k].shape[0]
    raise ValueError("Could not infer num_classes from checkpoint. "
                     "Your checkpoint might not include classifier weights.")


def load_finetuned_densenet161(model_path: str) -> nn.Module:
    ckpt = torch.load(model_path, map_location="cpu")

    # Case 1: checkpoint dict
    if isinstance(ckpt, dict):
        # Try common keys
        if "model_state" in ckpt:
            sd = ckpt["model_state"]
        elif "state_dict" in ckpt:
            sd = ckpt["state_dict"]
        else:
            # might already be a raw state_dict
            # detect by seeing tensor values
            if all(isinstance(v, torch.Tensor) for v in ckpt.values()):
                sd = ckpt
            else:
                sd = None

        if sd is not None:
            num_classes = infer_num_classes_from_state_dict(sd)
            model = build_densenet161_for_loading(num_classes)

            # Load (handle possible 'module.' prefix automatically by strict=False first)
            missing, unexpected = model.load_state_dict(sd, strict=False)

            # If it didn't load classifier due to mismatch, we'll warn,
            # but for feature extraction it's okay as long as backbone loads.
            if len(unexpected) > 0:
                print("[WARN] Unexpected keys:", unexpected[:10], "..." if len(unexpected) > 10 else "")
            if len(missing) > 0:
                print("[WARN] Missing keys:", missing[:10], "..." if len(missing) > 10 else "")

            return model

    # Case 2: entire model object saved
    if isinstance(ckpt, nn.Module):
        return ckpt

    raise ValueError("Unsupported checkpoint format. "
                     "Expected state_dict / dict checkpoint / saved nn.Module.")


# =========================
# Few-shot: build support prototypes
# =========================
def get_support_set(support_dir: str) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
    """
    Loads support set from folder with class subfolders:
      support_dir/
        ClassA/ img1.jpg ...
        ClassB/ ...
    Returns:
      X: [Ns, 3, H, W]
      y: [Ns] mapped to 0..Nway-1
      class_names: list of class folder names in index order
    """
    support_dir = Path(support_dir)
    class_folders = [p for p in support_dir.iterdir() if p.is_dir()]
    class_folders.sort(key=lambda x: x.name.lower())

    class_names = [p.name for p in class_folders]
    class_to_idx = {name: i for i, name in enumerate(class_names)}

    xs, ys = [], []
    for cf in class_folders:
        imgs = list_images(str(cf))
        if len(imgs) == 0:
            continue
        for img_path in imgs:
            xs.append(load_image(img_path))
            ys.append(class_to_idx[cf.name])

    X = torch.stack(xs) if xs else torch.empty(0)
    y = torch.tensor(ys, dtype=torch.long) if ys else torch.empty(0, dtype=torch.long)

    return X, y, class_names


# =========================
# Query parsing: map query image -> class index
# =========================
def normalize_name(s: str) -> str:
    """
    Normalize label names for matching:
    - lower
    - remove extension
    - remove trailing digits (e.g., "Buruli Ulcer1" -> "Buruli Ulcer")
    - collapse spaces
    """
    s = Path(s).stem
    s = s.lower().strip()
    s = re.sub(r"\d+$", "", s)          # strip trailing digits
    s = re.sub(r"\s+", " ", s).strip()  # normalize spaces
    return s


def build_query_labels(query_dir: str, class_names: List[str]) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
    """
    Loads query images from Query/*.jpg and assigns label by filename prefix.
    Example filename: "Buruli Ulcer1.jpg" -> label "Buruli Ulcer" -> match support class folder.
    Returns:
      Xq: [Nq, 3, H, W]
      yq: [Nq]
      files: list of filepaths (for reporting)
    """
    query_files = list_images(query_dir)

    # map normalized class name -> index
    norm_class_to_idx = {normalize_name(cn): i for i, cn in enumerate(class_names)}

    xs, ys, kept_files = [], [], []
    skipped = []

    for fp in query_files:
        base_norm = normalize_name(fp)
        if base_norm in norm_class_to_idx:
            xs.append(load_image(fp))
            ys.append(norm_class_to_idx[base_norm])
            kept_files.append(fp)
        else:
            skipped.append(fp)

    if skipped:
        print("\n[WARN] Skipped query images with unmatched labels (filename didn't match any support class):")
        for s in skipped[:20]:
            print("  -", s)
        if len(skipped) > 20:
            print(f"  ... (+{len(skipped)-20} more)")

    Xq = torch.stack(xs) if xs else torch.empty(0)
    yq = torch.tensor(ys, dtype=torch.long) if ys else torch.empty(0, dtype=torch.long)
    return Xq, yq, kept_files


# =========================
# Prototypical classifier
# =========================
@torch.no_grad()
def compute_prototypes(fe: nn.Module, Xs: torch.Tensor, ys: torch.Tensor, n_way: int) -> torch.Tensor:
    """
    Compute prototype per class: mean embedding of support images.
    """
    fe.eval()
    z = fe(Xs)  # [Ns, D]
    protos = []
    for c in range(n_way):
        protos.append(z[ys == c].mean(0))
    return torch.stack(protos, dim=0)  # [Nway, D]


@torch.no_grad()
def classify_queries(fe: nn.Module, protos: torch.Tensor, Xq: torch.Tensor, metric: str = "cosine") -> torch.Tensor:
    """
    Returns predicted class indices for queries.
    """
    fe.eval()
    zq = fe(Xq)  # [Nq, D]

    if metric == "cosine":
        zq = F.normalize(zq, dim=1)
        protos_n = F.normalize(protos, dim=1)
        logits = zq @ protos_n.t()  # [Nq, Nway]
    else:
        # negative squared euclidean distance as logits
        q2 = (zq**2).sum(1, keepdim=True)
        p2 = (protos**2).sum(1).unsqueeze(0)
        logits = -(q2 + p2 - 2 * (zq @ protos.t()))

    preds = logits.argmax(dim=1)
    return preds


def accuracy(preds: torch.Tensor, y: torch.Tensor) -> float:
    if y.numel() == 0:
        return float("nan")
    return (preds == y).float().mean().item()


# =========================
# MAIN
# =========================
def main():
    print("Device:", device)

    # 1) Load finetuned model
    model = load_finetuned_densenet161(MODEL_PATH)
    model.to(device)

    # 2) Freeze everything + eval
    for p in model.parameters():
        p.requires_grad = False
    model.eval()

    # 3) Wrap feature extractor
    fe = DenseNetFeatureExtractor(model).to(device).eval()

    # 4) Load support set
    Xs, ys, class_names = get_support_set(SUPPORT_DIR)
    if Xs.numel() == 0:
        raise RuntimeError(f"No support images found in: {SUPPORT_DIR}")
    n_way = len(class_names)
    print(f"Support loaded: {Xs.shape[0]} images | N-way={n_way} | classes={class_names}")

    # 5) Load query set + labels from filenames
    Xq, yq, qfiles = build_query_labels(QUERY_DIR, class_names)
    if Xq.numel() == 0:
        raise RuntimeError(f"No query images matched class names in: {QUERY_DIR}")
    print(f"Query loaded: {Xq.shape[0]} images (matched)")

    # Move to device
    Xs, ys = Xs.to(device), ys.to(device)
    Xq, yq = Xq.to(device), yq.to(device)

    # 6) Compute prototypes from support
    protos = compute_prototypes(fe, Xs, ys, n_way=n_way)

    # 7) Classify queries
    preds = classify_queries(fe, protos, Xq, metric="cosine")
    acc = accuracy(preds, yq)
    print(f"\nFew-shot accuracy (cosine prototypes): {acc*100:.2f}%")

    # 8) Optional: print per-image results (first 30)
    inv = {i: name for i, name in enumerate(class_names)}
    print("\nSample predictions:")
    for i in range(min(30, len(qfiles))):
        true_name = inv[int(yq[i].item())]
        pred_name = inv[int(preds[i].item())]
        fn = Path(qfiles[i]).name
        ok = "✅" if true_name == pred_name else "❌"
        print(f"{ok} {fn:35s}  true={true_name:20s}  pred={pred_name}")

if __name__ == "__main__":
    main()


Device: cpu
Support loaded: 6 images | N-way=3 | classes=['Buruli Ulcers', 'Post Kala Azar Dermal Leishmaniasis', 'Yaws']
Query loaded: 40 images (matched)

Few-shot accuracy (cosine prototypes): 60.00%

Sample predictions:
✅ Buruli Ulcers10.jpg                  true=Buruli Ulcers         pred=Buruli Ulcers
✅ Buruli Ulcers11.jpg                  true=Buruli Ulcers         pred=Buruli Ulcers
✅ Buruli Ulcers12.jpg                  true=Buruli Ulcers         pred=Buruli Ulcers
❌ Buruli Ulcers13.jpeg                 true=Buruli Ulcers         pred=Yaws
✅ Buruli Ulcers14.jpg                  true=Buruli Ulcers         pred=Buruli Ulcers
❌ Buruli Ulcers16.jpg                  true=Buruli Ulcers         pred=Yaws
✅ Buruli Ulcers17.png                  true=Buruli Ulcers         pred=Buruli Ulcers
❌ Buruli Ulcers18.jpg                  true=Buruli Ulcers         pred=Post Kala Azar Dermal Leishmaniasis
✅ Buruli Ulcers19.jpg                  true=Buruli Ulcers         pred=Buruli Ulcers
✅ Burul